# Feature Engineering: Activity: Steps Features

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')  # go up to scripts/ root
import os
from pathlib import Path

import pandas as pd
from datetime import datetime


from data_helper.clean_data_functions import clean_data_baseline_optimized, clean_data_no_data_days_optimized, drop_sleep_window_data, keep_timespan_only
from data_helper.download_REDCap_data import download_files_for_records
from plot_helper.descriptive_stats_plot import plot_descriptive_stats
from plot_helper.colors import COLORS


#dataframe for all features
df_step_features = pd.DataFrame()

## Download Data from REDCap
(Comment out if not needed)


In [ ]:
download_files_for_records(['all'], "venu3_step_device", "baseline_period_arm_1", "data/device_step")

download_files_for_records(['all'], "venu3_epoch", "baseline_period_arm_1", "data/epoch")

download_files_for_records(['all'], "venu3_daily_sum", "baseline_period_arm_1", "data/daily_summary")


## Create Base DataFrames


In [ ]:
#Get downloaded data
folder_device_step = Path("../../data/device_step")
folder_epoch = Path("../../data/epoch")

#Get merged csv file paths
step_files = [f.path for f in os.scandir(folder_device_step) if f.is_file() and f.name.endswith(".csv")]
epoch_files = [f.path for f in os.scandir(folder_epoch) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_step_raw = pd.concat(
    [pd.read_csv(f) for f in step_files],
    ignore_index=True
)

dt_utc_step = pd.to_datetime(
            df_step_raw["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_step_raw['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_step, df_step_raw["timezone"])
]


df_epoch_raw = pd.concat(
    [pd.read_csv(f) for f in epoch_files],
    ignore_index=True
)

dt_utc_epoch = pd.to_datetime(
            df_epoch_raw["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_epoch_raw['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_epoch, df_epoch_raw["timezone"])
]

step_shape_initial = df_step_raw.shape
epoch_shape_initial = df_epoch_raw.shape
#clean dataframes
##match baseline dates
print(f"Step data before cropping: {df_step_raw.shape}")
print(f"Epoch data before cropping: {df_epoch_raw.shape}")
df_step_raw, df_epoch_raw = clean_data_baseline_optimized([df_step_raw, df_epoch_raw])
print(f"Step data after cropping: {df_step_raw.shape}")
display(df_step_raw.head())
print(f"Epoch data after cropping: {df_epoch_raw.shape}")
display(df_epoch_raw.head())
print("------"*20)

##drop days with no data
df_step_raw, df_epoch_raw = clean_data_no_data_days_optimized([df_step_raw, df_epoch_raw])
print(f"Step data after dropping days with no data: {df_step_raw.shape}")
display(df_step_raw.head())
print(f"Epoch data after dropping days with no data: {df_epoch_raw.shape}")
display(df_epoch_raw.head())
print("------"*20)

#get sleep windows
sleep_window_file = "../../data/checks/data_sleep_windows_extraction_2026-07-08.csv"
df_sleep_windows = pd.read_csv(sleep_window_file)
display(df_sleep_windows.head())
n_sleep_windows = df_sleep_windows['study_id'].nunique()
print(f"Number of patients with sleep windows: {n_sleep_windows}")

#compare study ids across dataframes
study_ids_step = set(df_step_raw['study_id'].unique())
study_ids_epoch = set(df_epoch_raw['study_id'].unique())
study_ids_sleep = set(df_sleep_windows['study_id'].unique())
print(f"Study IDs in step data but not in sleep windows: {study_ids_step - study_ids_sleep}")
print(f"Study IDs in epoch data but not in sleep windows: {study_ids_epoch - study_ids_sleep}")

#exclude sleep window from step and epoch data
df_step_raw = drop_sleep_window_data(df_step_raw, df_sleep_windows)
df_epoch_raw = drop_sleep_window_data(df_epoch_raw, df_sleep_windows)

print("------"*20)
print(f"Step data after dropping sleep window data: {df_step_raw.shape}")
display(df_step_raw.head())
print(f"Epoch data after dropping sleep window data: {df_epoch_raw.shape}")
display(df_epoch_raw.head())

#drop from steps where total step < 200
df_step_agg = df_step_raw.groupby(['study_id', 'calendarDate'])['steps'].sum().reset_index()

df_dec_date_drop = df_step_agg[df_step_agg['steps'] < 200].copy()

mask = df_step_raw.set_index(["study_id", "calendarDate"]).index.isin(
    df_dec_date_drop.set_index(["study_id", "calendarDate"]).index
)

df_step_raw = df_step_raw[~mask]
print("------"*20)
print(f"Step data after dropping days with total steps < 200: {df_step_raw.shape}")
display(df_step_raw)

#drop those also from epoch data
mask_epoch = df_epoch_raw.set_index(["study_id", "calendarDate"]).index.isin(
    df_dec_date_drop.set_index(["study_id", "calendarDate"]).index
)
df_epoch_raw = df_epoch_raw[~mask_epoch]    
print(f"Epoch data after dropping days with total steps < 200: {df_epoch_raw.shape}")
display(df_epoch_raw)

#if no sleep window data, drop those study ids - calendarDate pairs from step and epoch data
dt = pd.to_datetime(df_sleep_windows['wake_up_time'], errors="coerce", utc=True)
df_sleep_windows['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt, df_sleep_windows['timezone'])
]

df_sleep_windows_agg = df_sleep_windows.groupby(['study_id', 'calendarDate']).size().reset_index(name='count')
df_step_raw = df_step_raw[df_step_raw.set_index(['study_id', 'calendarDate']).index.isin(df_sleep_windows_agg.set_index(['study_id', 'calendarDate']).index)]
print("------"*20)
print(f"Step data after dropping study_id - calendarDate pairs with no sleep window data: {df_step_raw.shape}")

df_epoch_raw = df_epoch_raw[df_epoch_raw.set_index(['study_id', 'calendarDate']).index.isin(df_sleep_windows_agg.set_index(['study_id', 'calendarDate']).index)]
print(f"Epoch data after dropping study_id - calendarDate pairs with no sleep window data: {df_epoch_raw.shape}")

#compare initial shape to final shape
print("------"*20)
print(f"Dropped {step_shape_initial[0] - df_step_raw.shape[0]} rows from step data")
print(f"Initial step data shape: {step_shape_initial}")
print(f"Final step data shape: {df_step_raw.shape}")
print(f"Dropped {epoch_shape_initial[0] - df_epoch_raw.shape[0]} rows from epoch data")
print(f"Initial epoch data shape: {epoch_shape_initial}")
print(f"Final epoch data shape: {df_epoch_raw.shape}")

print("------"*20)
n_step_raw = df_step_raw['study_id'].nunique()
print(f"Number of patients in step data: {n_step_raw}")
#epoch data
n_epoch_raw = df_epoch_raw['study_id'].nunique()
print(f"Number of patients in epoch data: {n_epoch_raw}")



## Total Steps

In [ ]:
df_step_f1 = df_step_raw[['study_id', 'calendarDate', 'steps']].copy()
display(df_step_f1.head())

df_step_f1 = df_step_f1.sort_values(by=['study_id']).reset_index(drop=True)

#for each date get total steps per day and study id
df_step_f1 = df_step_f1.groupby(['study_id', 'calendarDate'])['steps'].sum().reset_index()
display(df_step_f1.sort_values(by='steps', ascending=True).head())


#calculate descriptive stats for each study id
total_steps_stats = df_step_f1.groupby('study_id')['steps'].agg(
    mean_steps='mean',
    median_steps='median',
    std_steps='std',
    skewness_steps=('skew')  # positive = right skew
).reset_index()
display(total_steps_stats.head())

#create plots
fig_f1 = plot_descriptive_stats(
    df = df_step_f1,
    col_df = "steps", 
    df_descriptive_stats=total_steps_stats,
    col_mean="mean_steps",
    col_median="median_steps", 
    n_patients=n_step_raw,
    feature="Steps",
    title="Steps",
    colors = COLORS
    )

fig_f1.show()


#add steps stats to feature dataframe
df_step_features = total_steps_stats[['study_id', 'mean_steps', 'median_steps', 'std_steps']].copy()
display(df_step_features.head())

#create daily dataframe
df_step_features_daily  = df_step_f1[['study_id', 'calendarDate', 'steps']].copy()
df_step_features_daily = df_step_features_daily.rename(columns={'steps': 'daily_steps'})
display(df_step_features_daily.head())

## EPOCH


### Sedentary Duration

In [ ]:
#calculate descriptive stats for each study id
df_step_f2 = df_epoch_raw[['study_id', 'activeTimeInMs', 'intensity', 'datetime', 'datetime_utc', 'timezone', 'calendarDate']].copy()

#calculate total time spent in intensity=SEDENTARY for each day and study id
df_step_f2_filtered = df_step_f2.groupby(['study_id', 'calendarDate', 'intensity'])['activeTimeInMs'].sum().reset_index()

display(df_step_f2_filtered.head())

df_step_f2_filtered = df_step_f2_filtered.sort_values(by=['study_id', 'calendarDate']).reset_index(drop=True)

df_sedentary = df_step_f2_filtered[df_step_f2_filtered['intensity'] == 'SEDENTARY'].copy()

df_sedentary['activeTimeInMs'] = df_sedentary['activeTimeInMs'].fillna(0)
df_sedentary['activeTimeInHrs'] = df_sedentary['activeTimeInMs'].astype(float) / (1000 * 60 * 60)



display(df_sedentary.head())


sedentary_stats = df_sedentary.groupby('study_id')['activeTimeInHrs'].agg(
    mean_sedentary_time_h='mean',
    median_sedentary_time_h='median',
    std_sedentary_time_h='std',
).reset_index()

display(sedentary_stats.head())

#create plots
fig_f2 = plot_descriptive_stats(
    df = df_sedentary,
    col_df = "activeTimeInHrs",
    df_descriptive_stats=sedentary_stats,
    col_mean="mean_sedentary_time_h",
    col_median="median_sedentary_time_h",
    n_patients=n_epoch_raw,
    feature="Sedentary Duration",
    title="Sedentary Duration [h]",
    colors = COLORS

    )

fig_f2.show()

#add sedentary stats to feature dataframe
df_step_features = df_step_features.merge(
    sedentary_stats[['study_id', 'mean_sedentary_time_h', 'median_sedentary_time_h', 'std_sedentary_time_h']],
    on='study_id',
    how='outer'
)
display(df_step_features.head())

#create daily dataframe
#add daily sedentary to df_features_daily
df_step_features_daily = df_step_features_daily.merge(
    df_sedentary[['study_id', 'calendarDate', 'activeTimeInHrs']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_step_features_daily = df_step_features_daily.rename(columns={'activeTimeInHrs': 'daily_sedentary_time_h'})
display(df_step_features_daily.sort_values(by=['daily_sedentary_time_h'], ascending=False).head())

### Movement Duration

In [ ]:
#calculate descriptive stats for each study id
df_movement = df_step_f2_filtered[df_step_f2_filtered['intensity'] != 'SEDENTARY'].copy()
df_movement = df_movement.groupby(['study_id', 'calendarDate'])['activeTimeInMs'].sum().reset_index()

df_movement['activeTimeInMs'] = df_movement['activeTimeInMs'].fillna(0)
df_movement['activeTimeInHrs'] = df_movement['activeTimeInMs'].astype(float) / (1000 * 60 * 60)



display(df_movement.head())

movement_stats = df_movement.groupby('study_id')['activeTimeInHrs'].agg(
    mean_movement_time_h='mean',
    median_movement_time_h='median',
    std_movement_time_h='std',
).reset_index()

display(movement_stats.head())

#create plots
fig_f3 = plot_descriptive_stats(
    df = df_movement,
    col_df = "activeTimeInHrs",
    df_descriptive_stats=movement_stats,
    col_mean="mean_movement_time_h",
    col_median="median_movement_time_h",
    n_patients=n_epoch_raw,
    feature="Movement Duration",
    title="Movement Duration [h]",
    colors=COLORS

    )

fig_f3.show()


#add movement stats to feature dataframe
df_step_features = df_step_features.merge(
    movement_stats[['study_id', 'mean_movement_time_h', 'median_movement_time_h', 'std_movement_time_h']],
    on='study_id',
    how='outer'
)
display(df_step_features.head())

#create daily dataframe
#add daily MOVEMENT time to df_features_daily
df_step_features_daily = df_step_features_daily.merge(
    df_movement[['study_id', 'calendarDate', 'activeTimeInHrs']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_step_features_daily = df_step_features_daily.rename(columns={'activeTimeInHrs': 'daily_movement_time_h'})
display(df_step_features_daily.head())

### Movement / Sedentary ratio

In [ ]:
df_movement = df_movement.rename(columns={'activeTimeInMs': 'movement_time', 'activeTimeInHrs': 'movement_time_hrs'})
df_sedentary = df_sedentary.rename(columns={'activeTimeInMs': 'sedentary_time', 'activeTimeInHrs': 'sedentary_time_hrs'})
df_ratio = df_sedentary.merge(df_movement, on=['study_id', 'calendarDate'], how='outer')
#calculate ratio of sedentary to movement time
df_ratio['ratio'] = df_ratio['movement_time'] / df_ratio['sedentary_time']
display(df_ratio.head())



ratio_stats = df_ratio.groupby('study_id')['ratio'].agg(
    mean_ratio='mean',
    median_ratio='median',
    std_ratio='std',
).reset_index()

display(ratio_stats.head())

#create plots
fig_f4 = plot_descriptive_stats(
    df = df_ratio,
    col_df = "ratio",
    df_descriptive_stats=ratio_stats,
    col_mean="mean_ratio",
    col_median="median_ratio",
    n_patients=n_epoch_raw,
    feature="Movement/Sedentary Ratio",
    title="Ratio",
    bins = dict(start=0, end=1, size=0.1),
    colors=COLORS
    )

fig_f4.show()


#add ratio stats to feature dataframe
df_step_features = df_step_features.merge(
    ratio_stats[['study_id', 'mean_ratio', 'median_ratio', 'std_ratio']],
    on='study_id',
    how='outer'
)
display(df_step_features.head())

#create daily dataframe
#add daily ratio to df_features_daily
df_step_features_daily = df_step_features_daily.merge(
    df_ratio[['study_id', 'calendarDate', 'ratio']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_step_features_daily = df_step_features_daily.rename(columns={'ratio': 'daily_movement_sedentary_ratio'})
display(df_step_features_daily.head())

## Steps After Wake Up
### 2 hours after wake up (wake up time + 2h)

In [ ]:
df_step_f3 = df_step_raw[['study_id', 'datetime', 'steps', 'calendarDate', 'datetime_utc', 'timezone']].copy()
df_step_f3 = df_step_f3.sort_values(by=['study_id', 'datetime']).reset_index(drop=True)

df_wake_up_times = df_sleep_windows[['study_id', 'wake_up_time', 'timezone']].copy()


dt_utc = pd.to_datetime(
    df_wake_up_times["wake_up_time"],
    utc=True,
    errors="coerce"
)

df_wake_up_times["wake_up_time"] = [
    ts.tz_convert(tz)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_wake_up_times["timezone"])
]

df_wake_up_times["wake_up_time_2h"] = [
    ts.tz_convert(tz) + pd.Timedelta(hours=2)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_wake_up_times["timezone"])
]


df_wake_up_times['wake_up_window'] = df_wake_up_times.groupby('study_id').cumcount() + 1
display(df_wake_up_times.head())

#keep only steps between wake_up_time and wake_up_time + 2h for each study id
df_step_f3_filtered = keep_timespan_only(df_step_f3, df_wake_up_times, window_col="wake_up_window", start_col='wake_up_time', end_col='wake_up_time_2h')
display(df_step_f3_filtered.head())

#for each date get total steps per day and study id
df_step_f3_filtered = df_step_f3_filtered.groupby(['study_id', 'calendarDate', 'wake_up_window'])['steps'].sum().reset_index()
display(df_step_f3_filtered.head())

#calculate descriptive stats for each study id
steps_2h_after_wake_up = df_step_f3_filtered.groupby('study_id')['steps'].agg(
    mean_steps_2h='mean',
    median_steps_2h='median',
    std_steps_2h='std',
).reset_index()
display(steps_2h_after_wake_up.head())

#create plots
fig_f5 = plot_descriptive_stats(
    df = df_step_f3_filtered,
    col_df = "steps",
    df_descriptive_stats=steps_2h_after_wake_up,
    col_mean="mean_steps_2h",
    col_median="median_steps_2h",
    n_patients=n_step_raw,
    feature="Steps 2h after Wake Up",
    title="Steps 2h after Wake Up",
    colors = COLORS

    )

fig_f5.show()


#add steps 2h after wake up stats to feature dataframe
df_step_features = df_step_features.merge(
    steps_2h_after_wake_up[['study_id', 'mean_steps_2h', 'median_steps_2h', 'std_steps_2h']],
    on='study_id',
    how='outer'
)
display(df_step_features.head())

#create daily dataframe
#add daily steps 2h after wake up to df_features_daily
df_step_features_daily = df_step_features_daily.merge(
    df_step_f3_filtered[['study_id', 'calendarDate', 'steps']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_step_features_daily = df_step_features_daily.rename(columns={'steps': 'daily_steps_2h_after_wake_up'})
display(df_step_features_daily.head())

### 4 hours after wake up (2h after wake up time + 2h)

In [ ]:
dt_utc = pd.to_datetime(
    df_wake_up_times["wake_up_time"],
    utc=True,
    errors="coerce"
)

df_wake_up_times["wake_up_time"] = [
    ts.tz_convert(tz)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_wake_up_times["timezone"])
]

df_wake_up_times["wake_up_time_4h"] = [
    ts.tz_convert(tz) + pd.Timedelta(hours=4)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_wake_up_times["timezone"])
]

df_wake_up_times["wake_up_time_2h"] = [
    ts.tz_convert(tz) + pd.Timedelta(hours=2)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_wake_up_times["timezone"])
]


display(df_wake_up_times.head())
#keep only steps between wake_up_time and wake_up_time + 2h for each study id

df_step_f3_filtered_2 = keep_timespan_only(df_step_f3, df_wake_up_times, window_col="wake_up_window", start_col='wake_up_time_2h', end_col='wake_up_time_4h')
display(df_step_f3_filtered_2.head())

#for each date get total steps per day and study id
df_step_f3_filtered_2 = df_step_f3_filtered_2.groupby(['study_id', 'calendarDate', 'wake_up_window'])['steps'].sum().reset_index()
display(df_step_f3_filtered_2.head())

#calculate descriptive stats for each study id
steps_4h_after_wake_up = df_step_f3_filtered_2.groupby('study_id')['steps'].agg(
    mean_steps_4h='mean',
    median_steps_4h='median',
    std_steps_4h='std',
).reset_index()
display(steps_4h_after_wake_up.head())

#create plots
fig_f6 = plot_descriptive_stats(
    df = df_step_f3_filtered_2,
    col_df = "steps",
    df_descriptive_stats=steps_4h_after_wake_up,
    col_mean="mean_steps_4h",
    col_median="median_steps_4h",
    n_patients=n_step_raw,
    feature="Steps 4h after Wake Up",
    title="Steps 4h after Wake Up",
    colors=COLORS
    )

fig_f6.show()


#add steps 4h after wake up stats to feature dataframe
df_step_features = df_step_features.merge(
    steps_4h_after_wake_up[['study_id', 'mean_steps_4h', 'median_steps_4h', 'std_steps_4h']],
    on='study_id',
    how='outer'
)
display(df_step_features.head())

#create daily dataframe
#add daily steps 4h after wake up to df_features_daily
df_step_features_daily = df_step_features_daily.merge(
    df_step_f3_filtered_2[['study_id', 'calendarDate', 'steps']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_step_features_daily = df_step_features_daily.rename(columns={'steps': 'daily_steps_4h_after_wake_up'})
display(df_step_features_daily.head())

## Steps Before Sleep Onset
### 2h before Sleep Onset (Onset - 2h to Onset)

In [ ]:
df_step_f4 = df_step_raw[['study_id', 'datetime', 'steps', 'datetime_utc', 'timezone', 'calendarDate']].copy()
df_step_f4 = df_step_f4.sort_values(by=['study_id', 'datetime']).reset_index(drop=True)

df_onset_times = df_sleep_windows[['study_id', 'onset_time', 'timezone']].copy()
dt_utc = pd.to_datetime(
    df_onset_times["onset_time"],
    utc=True,
    errors="coerce"
)

df_onset_times["onset_time"] = [
    ts.tz_convert(tz)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_onset_times["timezone"])
]

df_onset_times["onset_time_2h"] = [
    ts.tz_convert(tz) - pd.Timedelta(hours=2)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_onset_times["timezone"])
]

df_onset_times['onset_window'] = df_onset_times.groupby('study_id').cumcount() + 1
display(df_onset_times[df_onset_times['study_id']=='DEC_01'].head())

#keep only steps between onset_time - 2h and onset_time for each study id
df_step_f4_filtered = keep_timespan_only(df_step_f4, df_onset_times, window_col="onset_window", start_col='onset_time_2h', end_col='onset_time')
display(df_step_f4_filtered.head())

#for each date get total steps per day and study id
df_step_f4_filtered = df_step_f4_filtered.groupby(['study_id', 'calendarDate', 'onset_window'])['steps'].sum().reset_index()
display(df_step_f4_filtered.head())

#calculate descriptive stats for each study id
steps_2h_before_onset = df_step_f4_filtered.groupby('study_id')['steps'].agg(
    mean_steps_onset_2h='mean',
    median_steps_onset_2h='median',
    std_steps_onset_2h='std',
).reset_index()
display(steps_2h_before_onset.head())

#create plots
fig_f7 = plot_descriptive_stats(
    df = df_step_f4_filtered,
    col_df = "steps",
    df_descriptive_stats=steps_2h_before_onset,
    col_mean="mean_steps_onset_2h",
    col_median="median_steps_onset_2h",
    n_patients=n_step_raw,
    feature="Steps 2h before Onset",
    title="Steps 2h before Onset",
    bins=dict(start=df_step_f4_filtered['steps'].min(), end=df_step_f4_filtered['steps'].max(), size=(df_step_f4_filtered['steps'].max() - df_step_f4_filtered['steps'].min()) / 24),
    bins_stats = dict(start=0, end=round(steps_2h_before_onset['mean_steps_onset_2h'].max()+150, -3), size=150),
    colors=COLORS
    )

fig_f7.show()


#add steps 2h after wake up stats to feature dataframe
df_step_features = df_step_features.merge(
    steps_2h_before_onset[['study_id', 'mean_steps_onset_2h', 'median_steps_onset_2h', 'std_steps_onset_2h']],
    on='study_id',
    how='outer'
)
display(df_step_features.head())

#create daily dataframe
#add daily steps 2h before onset to df_features_daily
df_step_features_daily = df_step_features_daily.merge(
    df_step_f4_filtered[['study_id', 'calendarDate', 'steps']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_step_features_daily = df_step_features_daily.rename(columns={'steps': 'daily_steps_2h_before_onset'})
display(df_step_features_daily.head())

### 4h before Sleep Onset (Onset - 4h to Onset - 2h)

In [ ]:
df_onset_times['onset_time_4h'] = pd.to_datetime(df_onset_times['onset_time_2h']) - pd.Timedelta(hours=2)

dt_utc = pd.to_datetime(
    df_onset_times["onset_time"],
    utc=True,
    errors="coerce"
)

df_onset_times["onset_time"] = [
    ts.tz_convert(tz)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_onset_times["timezone"])
]

df_onset_times["onset_time_4h"] = [
    ts.tz_convert(tz) - pd.Timedelta(hours=4)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_onset_times["timezone"])
]

df_onset_times["onset_time_2h"] = [
    ts.tz_convert(tz) - pd.Timedelta(hours=2)
    if pd.notna(ts) and pd.notna(tz)
    else pd.NaT
    for ts, tz in zip(dt_utc, df_onset_times["timezone"])
]
display(df_onset_times[df_onset_times['study_id']=='DEC_01'].head())
#keep only steps between wake_up_time and wake_up_time + 2h for each study id
df_step_f4_filtered_2 = keep_timespan_only(df_step_f4, df_onset_times, window_col="onset_window", start_col='onset_time_4h', end_col='onset_time_2h')
display(df_step_f4_filtered_2.head())

#for each date get total steps per day and study id
df_step_f4_filtered_2 = df_step_f4_filtered_2.groupby(['study_id', 'calendarDate', 'onset_window'])['steps'].sum().reset_index()
display(df_step_f4_filtered_2.head())

#calculate descriptive stats for each study id
steps_4h_after_onset = df_step_f4_filtered_2.groupby('study_id')['steps'].agg(
    mean_steps_onset_4h='mean',
    median_steps_onset_4h='median',
    std_steps_onset_4h='std',
).reset_index()
display(steps_4h_after_onset.head())

#create plots
fig_f8 = plot_descriptive_stats(
    df = df_step_f4_filtered_2,
    col_df = "steps",
    df_descriptive_stats=steps_4h_after_onset,
    col_mean="mean_steps_onset_4h",
    col_median="median_steps_onset_4h",
    n_patients=n_step_raw,
    feature="Steps 4h before Onset",
    title="Steps 4h before Onset",
    colors=COLORS
    # bins=dict(start=df_step_f4_filtered_2['steps'].min(), end=df_step_f4_filtered_2['steps'].max(), size=(df_step_f4_filtered_2['steps'].max() - df_step_f4_filtered_2['steps'].min()) / 24),
    # bins_stats = dict(start=0, end=round(steps_4h_after_onset['mean_steps_onset_4h'].max()+150, -3), size=150),

    )

fig_f8.show()



#add steps 4h after wake up stats to feature dataframe
df_step_features = df_step_features.merge(
    steps_4h_after_onset[['study_id', 'mean_steps_onset_4h', 'median_steps_onset_4h', 'std_steps_onset_4h']],
    on='study_id',
    how='outer'
)
display(df_step_features.head())

#create daily dataframe
#add daily steps 4h after wake up to df_features_daily
df_step_features_daily = df_step_features_daily.merge(
    df_step_f4_filtered_2[['study_id', 'calendarDate', 'steps']],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_step_features_daily = df_step_features_daily.rename(columns={'steps': 'daily_steps_4h_before_onset'})
display(df_step_features_daily.head())

## Store df_feature

In [ ]:
#if there already exists a file with same name move it to subfolder "archive"
if not os.path.exists('../../output/1_feature_extraction/archive'):
    os.makedirs('../../output/1_feature_extraction/archive')
files = os.listdir('../../output/1_feature_extraction')
for file in files:
    if file.startswith('df_features_step_') and file.endswith('.csv'):
        os.rename(f'../../output/1_feature_extraction/{file}', f'../../output/1_feature_extraction/archive/{file}')
    elif file.startswith('df_features_daily_step_') and file.endswith('.csv'):
        os.rename(f'../../output/1_feature_extraction/{file}', f'../../output/1_feature_extraction/archive/{file}')

#store df_feature as csv
date = datetime.now().strftime("%Y-%m-%d")
df_step_features.to_csv(f'../../output/1_feature_extraction/df_features_step_{date}.csv', index=False)
df_step_features_daily.to_csv(f'../../output/1_feature_extraction/df_features_daily_step_{date}.csv', index=False)